# Task 1 / Task 3 blends without the zero ratings

The leaderboard leaves out ratings of 0 (lecture 6 Nov: *"the leaderboard does not include the zero ratings for the assignment"*;
our public Task 1 score 0.5618 matches the out-of-fold MSE on the non-zero validation rows, 0.566, not the 0.708 on all rows).
So the blends are fit and scored on the validation rows with a rating > 0 (`fit_on_nonzero_ratings=True` in `lib/config.py`), and
predict from 1 to 5. The sub-models stay as stored (no retraining); the zero model's P(rating = 0) stays a blend input.

**Run All** (about a minute per task). The current prediction files are copied to `output/before_zero_fix/` first.

In [ ]:
import shutil
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))               # assignment1.py and lib/ live next to this notebook (in submission/)

import numpy as np

import assignment1
import lib.config as model_configs
from lib.helper_functioins import mse
from lib.model_storage import load_model_result, trained_model_path
from lib.models.blender import store_blender

backup_dir = assignment1.TASK1_PREDICTION_FILE.parent / "before_zero_fix"
backup_dir.mkdir(exist_ok=True)
for prediction_file in (assignment1.TASK1_PREDICTION_FILE, assignment1.TASK3_PREDICTION_FILE):
    if prediction_file.exists() and not (backup_dir / prediction_file.name).exists():
        shutil.copy2(prediction_file, backup_dir / prediction_file.name)
print("backups:", sorted(path.name for path in backup_dir.iterdir()))


# refits ridge + LightGBM on the non-zero validation rows, stores both, writes the submitted method's predictions
def reblend(task_name: str, task, blender_configs, submitted_method: str, train_blender, write_predictions):
    nonzero = task.validation_data.ratings > 0
    ratings = task.validation_data.ratings[nonzero]
    inputs = {name: load_model_result(task_name, name) for name in blender_configs[0].blended_models}
    old_ridge = load_model_result(task_name, "ridge_blender")      # the blend fit on all rows (stored by the retraining run)
    print(f"{task_name}: {nonzero.sum()} of {len(nonzero)} validation rows have a rating > 0")
    print(f"  before: ridge blend fit on all rows   -> MSE on rows > 0 {mse(old_ridge.valid_predictions[nonzero], ratings):.5f}")
    scores = {}
    for blender_config in blender_configs:
        blend = train_blender(task, inputs, blender_config)
        store_blender(blend, task_name)
        scores[blender_config.method] = mse(blend.valid_predictions[nonzero], ratings)
        print(f"  now:    {blender_config.method:<8} blend fit on rows > 0 -> out-of-fold MSE on rows > 0 {scores[blender_config.method]:.5f}"
              f" | test predictions {blend.test_predictions.min():.2f} - {blend.test_predictions.max():.2f}")
        if blender_config.method == submitted_method:
            write_predictions(blend.test_predictions)
    print(f"  wrote the {submitted_method} blend (TASK*_SUBMITTED_BLENDER); best out-of-fold: {min(scores, key=scores.get)}")

In [ ]:
task1 = assignment1.load_task1()
reblend("task1", task1, (model_configs.TASK1_RIDGE_BLENDER, model_configs.TASK1_LIGHTGBM_BLENDER),
        model_configs.TASK1_SUBMITTED_BLENDER.method, assignment1.train_task1_blender, assignment1.write_task1_predictions)
del task1

In [ ]:
task3 = assignment1.load_task3()
reblend("task3", task3, (model_configs.TASK3_RIDGE_BLENDER, model_configs.TASK3_LIGHTGBM_BLENDER),
        model_configs.TASK3_SUBMITTED_BLENDER.method, assignment1.train_task3_blender, assignment1.write_task3_predictions)
del task3